# Baseline SleepFM sur PhysioNet 2018 — Google Colab

Stadification (5 classes : Wake, N1, N2, N3, REM) avec le checkpoint officiel
SleepFM, sur de vrais enregistrements PhysioNet 2018.

**Avant de lancer** : `Exécution > Modifier le type d'exécution > GPU T4`, puis
`Exécution > Tout exécuter`.

Les sujets sont traités **par lots** : téléchargement → extraction → embeddings,
puis le brut est supprimé et seuls les embeddings (~5 Mo/sujet) sont gardés.
Avec Google Drive activé (cellule 3), une déconnexion Colab ne fait rien perdre :
relancer reprend là où on s'était arrêté.

Durée indicative (l'extraction CPU domine) : ~1 h 15 pour 100 sujets,
~2 h 30 pour 200.

PhysioNet 2018 est en accès libre (pas de DUA), donc pas de restriction à
l'utiliser sur Colab — contrairement à MESA/SHHS/CHAT (NSRR).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "Pas de GPU : Exécution > Modifier le type d'exécution > GPU T4"
!df -h /content | tail -1

## 1. Code et dépendances

In [ ]:
%cd /content
!git clone -q https://github.com/sumarc183-design/sleepfm-pipeline.git || (cd sleepfm-pipeline && git pull -q)
%cd /content/sleepfm-pipeline
!pip install -q -r requirements.txt

## 2. Checkpoint officiel SleepFM (`best.pt`, ~84 Mo, versionné dans le dépôt officiel)

In [ ]:
![ -d third_party/sleepfm ] || git clone -q --depth 1 https://github.com/rthapa84/sleepfm-codebase.git third_party/sleepfm
!ls -lh third_party/sleepfm/sleepfm/checkpoint/best.pt

## 3. Paramètres

- `N_SUBJECTS` : les N premiers sujets d'entraînement PhysioNet 2018 (994 au
  total). Le résultat du README (κ = 0,584) a été obtenu avec 20.
- `USE_DRIVE` : garde les embeddings sur Google Drive (~5 Mo/sujet) pour
  reprendre après une déconnexion. Colab demande l'autorisation d'accès au Drive.

In [ ]:
N_SUBJECTS = 100
USE_DRIVE = True

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    EMB_DIR = "/content/drive/MyDrive/sleepfm_embeddings/physionet2018"
else:
    EMB_DIR = "/content/embeddings/physionet2018"
print("Embeddings :", EMB_DIR)

## 4. Baseline : téléchargement + extraction par lots → embeddings (GPU) → 5 sondes

Split **80/20 par sujet** (jamais par époque). Cinq sondes sur les mêmes
embeddings : régression logistique (baseline, protocole officiel) et sa
version pondérée, XGBoost (GPU), MLP, et un BiGRU qui lit la nuit entière
(contexte temporel). Pour chacune : accuracy, kappa, F1 macro, ROC-AUC, et
le **gap** train − test (un gap élevé = surapprentissage). Sous le tableau :
MLP ou XGBoost (XGBoost seulement s'il gagne de plus de 0,02 en F1 macro).
Comptez ~5 à 10 min pour les sondes ; `--probes logistic` pour la baseline
seule.

Si les embeddings sont déjà sur Drive (run précédent), seule cette étape de
sondes est refaite.

In [ ]:
!python run_baseline.py \
    --physionet-subjects {N_SUBJECTS} \
    --embeddings-dir "{EMB_DIR}" \
    --checkpoint third_party/sleepfm/sleepfm/checkpoint/best.pt \
    --num-threads 2 \
    --chunk-size 8 \
    --batch-size 128 2>&1 | grep -v -i warning | tee results_physionet2018_{N_SUBJECTS}.txt

## 5. Garder le résultat

Télécharger `results_physionet2018_<N>.txt` (panneau Fichiers, ou la cellule
ci-dessous).

**Lecture du chiffre** : la régression logistique sur embeddings figés est le
protocole officiel, c'est elle qui se compare à la publication ; RF et
XGBoost disent si une frontière non linéaire apporte quelque chose sur les
mêmes embeddings. Le papier SleepFM rapporte ses chiffres de
stadification sur MESA/SHHS, pas sur PhysioNet 2018 : ce kappa est une
référence interne, la comparaison stricte au chiffre publié se fera sur MESA.

In [ ]:
from google.colab import files
files.download(f"/content/sleepfm-pipeline/results_physionet2018_{N_SUBJECTS}.txt")